# Reflection in agentic AI: give the reviewer something to check
**Alireza Akhavan · [Agentic_AI](https://github.com/Alireza-Akhavan/Agentic_AI)**

A reflection loop gives a first attempt a second pass against the original goal. Here one function orchestrates three tasks: chart design, SQL generation and essay writing. The roles stay the same; the evidence changes.

**Run it:** Use a Python 3 Jupyter kernel with `matplotlib` installed (`%pip install matplotlib` if needed), then Run All. The default replay uses hand-authored examples, not recorded model responses. It needs no keys, external data or network. For live generation install `openai`, set `OPENAI_API_KEY` outside the notebook, and switch `LIVE` below. A full live run makes nine paid calls and sends the examples and rendered draft chart to the provider.

Adapted from the visual generation, SQL generation and essay workflows in this repository, based on [DeepLearning.AI’s Agentic AI course](https://www.deeplearning.ai/courses/agentic-ai/).

In [ ]:
import base64, io, json, sqlite3
from IPython.display import display, Image
import matplotlib.pyplot as plt

LIVE = False
MODEL = 'gpt-4.1-mini'  # use a vision-capable model for the chart reviewer
if LIVE:
    from openai import OpenAI
    client = OpenAI()


def ask(prompt, replay, png=None):
    if not LIVE:
        return replay
    content = [{'type': 'text', 'text': prompt}]
    if png is not None:
        url = 'data:image/png;base64,' + base64.b64encode(png).decode()
        content.append({'type': 'image_url', 'image_url': {'url': url}})
    return client.chat.completions.create(model=MODEL, messages=[
        {'role': 'user', 'content': content}]).choices[0].message.content.strip()


def reflect(goal, inspect, examples):
    first, critique, second = examples
    draft = ask(goal, first)
    evidence, image = inspect(draft)
    context = f'Goal: {goal}\nDraft: {draft}\nEvidence: {evidence}'
    review = ask(context + '\nIdentify a concrete mismatch and suggest a fix. If none, say so.', critique, image)
    revised = ask(context + f'\nReview: {review}\nReturn only the revised artifact in the requested format.', second)
    for name, value in [('Draft', draft), ('Evidence', evidence), ('Review', review), ('Revision', revised)]:
        print(f'{name}: {value}\n')
    return revised

## Chart: can you tell what the bars measure?
The model proposes a small JSON chart specification rather than executable Python. A fixed renderer owns the numbers. The reviewer gets the rendered PNG as well as the spec, so it can judge what a reader actually sees.

In [ ]:
sales = {'2024': 1200, '2025': 1500}  # illustrative Q1 coffee revenue, USD

def chart(spec):
    s = json.loads(spec)
    fig, ax = plt.subplots(figsize=(5, 3), layout='constrained')
    ax.bar(list(sales), list(sales.values()), color='#39796b')
    ax.set(title=s['title'], ylabel=s['ylabel'])
    buf = io.BytesIO()
    fig.savefig(buf, format='png'); plt.close(fig)
    png = buf.getvalue()
    display(Image(data=png))
    return 'Rendered from fixed Q1 revenue: ' + str(sales) + ' USD.', png

spec = reflect('Compare Q1 coffee revenue for 2024 and 2025 in USD. Return bare JSON with string keys title and ylabel only.', chart, (
    '{"title":"Coffee sales","ylabel":"Sales"}',
    'The bars compare years, but the reader cannot tell the period or whether sales means dollars or cups.',
    '{"title":"Q1 coffee revenue by year","ylabel":"Revenue (USD)"}'))
chart(spec);

## SQL: a successful query can still be wrong
Each row is an inventory event. Restocks move units but earn no revenue. In this small fixture we can check the answer by hand: blue sells 3 × $5 and red sells 2 × $6. The reviewer sees the returned rows, not just plausible SQL.

In [ ]:
db = sqlite3.connect(':memory:')
db.execute('CREATE TABLE events(color TEXT, kind TEXT, units INTEGER, price_usd INTEGER)')
rows = [('blue','sale',3,5), ('red','sale',2,6), ('red','restock',10,6)]
db.executemany('INSERT INTO events VALUES (?,?,?,?)', rows)
db.execute('PRAGMA query_only=ON')

def query(sql):
    if not sql.lstrip().upper().startswith('SELECT '):
        return 'Expected one SELECT statement.', None
    try:
        return repr(db.execute(sql).fetchmany(5)), None
    except sqlite3.Error as err:
        return str(err), None

draft_sql = 'SELECT color, SUM(units*price_usd) FROM events GROUP BY color ORDER BY 2 DESC LIMIT 1'
goal = ('Which color earns the most sale revenue? Return only a SQLite SELECT statement yielding color and revenue. '
        'Schema: events(color, kind, units, price_usd). Rows: ' + repr(rows))
sql = reflect(goal, query, (draft_sql,
    'Red at $72 includes $60 of restocks. Filter to sales before grouping.',
    draft_sql.replace('GROUP BY', "WHERE kind='sale' GROUP BY")))
result, _ = query(sql)
print('Revised result:', result, '| Fixture check:', 'PASS' if result == repr([('blue', 15)]) else 'FAIL')

## Essay: critique the argument, not just the prose
Unlike the SQL result, an essay has no single expected output. Give the reviewer a rubric. Here a useful revision must engage with an objection, not just sound more confident.

In [ ]:
rubric = 'Look for a clear claim, a substantive objection and a response. Flag unsupported factual claims.'
essay = reflect('Write a short argument about whether governments should regulate social media. Address one objection.',
    lambda draft: (rubric, None), (
    'Governments should regulate social media. Platforms affect public debate, so oversight matters.',
    'Oversight is too broad. Name one intervention and address the risk that officials could police speech.',
    'Governments should require platforms to explain moderation decisions and offer an appeal. Critics rightly worry that oversight could let officials dictate which opinions stay online. A right to reasons and review would regulate the process, not prescribe which views are acceptable.'))

## What to take away
A second call is not a correctness check. In the SQL example, removing the restock row would let the flawed query pass by coincidence. In the chart example, the source numbers can be right while the label misleads. In the essay example, a rubric can expose a missing objection but cannot verify a factual claim.

**Try it:** Add a larger blue restock and rerun the SQL cell. Does the first query now pick the right *color* for the wrong reason? Compare the revenue too. For an open-ended task, inspect both drafts yourself before accepting the revision.